<a href="https://colab.research.google.com/github/pranavvup-byte/ml-pratical-week/blob/main/ml_week_8_pra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/collab/placement_predict_50k_adjusted (1).csv')

print(df.head())
print("Shape:", df.shape)


In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# ---------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------
RANDOM_STATE = 42
TARGET_COL = "PlacementStatus"
DATA_PATH = "/content/drive/MyDrive/collab/placement_predict_50k_adjusted (1).csv"


# =========================================================
# STEP 1: LOAD + PREPROCESS THE DATA
# =========================================================
df = pd.read_csv(DATA_PATH)

print("First 5 rows:")
print(df.head())
print("\nShape:", df.shape)

# Drop the anomaly flag if it exists
if "IsAnomaly" in df.columns:
    df = df.drop(columns=["IsAnomaly"])

# Target and features
y = df[TARGET_COL].astype(int)
X = df.drop(columns=[TARGET_COL])

# Identify categorical and numerical columns
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

print("\nCategorical columns:", cat_cols)
print("Numerical columns:", num_cols)

# Label encode categorical columns
encoders = {}

for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))
    encoders[c] = le

# Fill missing numerical values using median
imputer = SimpleImputer(strategy="median")
X[num_cols] = imputer.fit_transform(X[num_cols])

# Standardize numerical columns
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

print("\nPreprocessing completed successfully.")


# =========================================================
# STEP 2: TRAIN / VALIDATION / TEST SPLIT
# =========================================================
VAL_SIZE = 0.15
TEST_SIZE = 0.15

# First split the test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

# Then split train and validation sets
val_ratio = VAL_SIZE / (1 - TEST_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=val_ratio,
    stratify=y_train_val,
    random_state=RANDOM_STATE
)

print("\nData Split:")
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")


# =========================================================
# PART 1: ADABOOST
# =========================================================
print("\nPART 1: ADABOOST")
print("-" * 45)

# Weak learner: shallow decision tree
ada_base = DecisionTreeClassifier(
    max_depth=2,
    random_state=RANDOM_STATE
)

ada = AdaBoostClassifier(
    estimator=ada_base,
    n_estimators=200,
    learning_rate=0.5,
    random_state=RANDOM_STATE
)

# Train AdaBoost
t0 = time.time()

ada.fit(X_train, y_train)

ada_fit_time = time.time() - t0

# Predictions
ada_val_pred = ada.predict(X_val)
ada_val_proba = ada.predict_proba(X_val)[:, 1]

# Evaluation
ada_accuracy = accuracy_score(y_val, ada_val_pred)
ada_f1 = f1_score(y_val, ada_val_pred)
ada_roc_auc = roc_auc_score(y_val, ada_val_proba)

print("Accuracy:", round(ada_accuracy, 4))
print("F1 Score:", round(ada_f1, 4))
print("ROC-AUC:", round(ada_roc_auc, 4))
print("Number of Trees:", ada.n_estimators)
print("Fit Time:", round(ada_fit_time, 2), "seconds")


# =========================================================
# PART 2: XGBOOST
# =========================================================
print("\nPART 2: XGBOOST")
print("-" * 45)

xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Train XGBoost with early stopping
t0 = time.time()

xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_fit_time = time.time() - t0

# Predictions
xgb_val_pred = xgb.predict(X_val)
xgb_val_proba = xgb.predict_proba(X_val)[:, 1]

# Evaluation
xgb_accuracy = accuracy_score(y_val, xgb_val_pred)
xgb_f1 = f1_score(y_val, xgb_val_pred)
xgb_roc_auc = roc_auc_score(y_val, xgb_val_proba)

print("Accuracy:", round(xgb_accuracy, 4))
print("F1 Score:", round(xgb_f1, 4))
print("ROC-AUC:", round(xgb_roc_auc, 4))
print("Trees Used:", xgb.best_iteration + 1)
print("Fit Time:", round(xgb_fit_time, 2), "seconds")


# =========================================================
# PART 3: COMPARE ADABOOST AND XGBOOST
# =========================================================
print("\nPART 3: MODEL COMPARISON")
print("-" * 45)

results = [
    {
        "Model": "AdaBoost",
        "Validation Accuracy": ada_accuracy,
        "Validation F1": ada_f1,
        "Validation ROC-AUC": ada_roc_auc,
        "Trees Used": ada.n_estimators,
        "Fit Time (sec)": round(ada_fit_time, 2)
    },
    {
        "Model": "XGBoost",
        "Validation Accuracy": xgb_accuracy,
        "Validation F1": xgb_f1,
        "Validation ROC-AUC": xgb_roc_auc,
        "Trees Used": xgb.best_iteration + 1,
        "Fit Time (sec)": round(xgb_fit_time, 2)
    }
]

leaderboard = pd.DataFrame(results)

leaderboard = leaderboard.sort_values(
    "Validation Accuracy",
    ascending=False
).reset_index(drop=True)

print(leaderboard.to_string(index=False))


# ---------------------------------------------------------
# STEP 4: SAVE RESULTS
# ---------------------------------------------------------
leaderboard.to_csv(
    "boosting_benchmark_results.csv",
    index=False
)

print("\nSaved results -> boosting_benchmark_results.csv")
